# BanglaHalluEval Summarization Hallucination Pipeline
This notebook generates 3 hallucinated summaries per document using GPT-5.4.
Patterns: Intrinsic, Non-factual, Factual Contradiction.


In [ ]:
!pip -q install openai python-dotenv

In [ ]:
import os
import csv
from pathlib import Path
from openai import OpenAI


## Configuration
Set your API key and paths below.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BanglaHalluEval'
Path(BASE_DIR).mkdir(parents=True, exist_ok=True)
print('Saving outputs to:', BASE_DIR)


In [ ]:
os.environ['OPENAI_API_KEY'] = ''  # <- paste your key
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4')
OUTPUT_CSV = f"{BASE_DIR}/summarization_hallucinations_pilot_20.csv"
DOCUMENT_COLUMN = 'question'
SUMMARY_COLUMN = 'summary'
CHECKPOINT_EVERY = 50


In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No CSV uploaded. Please upload a CSV file.')

INPUT_CSV = next(iter(uploaded.keys()))
INPUT_CSV = f"/content/{INPUT_CSV}"
print('Using input:', INPUT_CSV)


In [ ]:
def load_rows(csv_path):
    with open(csv_path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def write_rows(csv_path, rows, fieldnames):
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def build_prompt(template, document, right_summary):
    return template.replace('<Here is the test question>', document).replace(
        '<Here is the right summary of the test question>', right_summary
    )


def normalize_summary(text):
    cleaned = (text or '').strip()
    if not cleaned:
        return cleaned
    marker = '#Hallucinated Summary#'
    if marker in cleaned:
        cleaned = cleaned.split(marker, 1)[-1].strip()
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    return lines[0] if lines else cleaned


In [ ]:
prompt_1 = (
    "I want you to act as a Hallucinated Summary (HS) generator. The answer should be given in BANGLA. "
    "Given a Question (Q) and the Right Summary (RS), your objective is to write a Hallucinated Summary (HS) "
    "that sounds plausible but is factually incorrect.\n\n"
    "You MUST use the following hallucination method:\n"
    "The summary introduces information that is either too specific "
    "(adds details, numbers, names, or conclusions not present in the source) or too vague "
    "(omits critical specifics that the source clearly states). The hallucination must arise "
    "from within the source context but distort or over-extend it.\n\n"
    "Example —\n"
    "#Question (Q)#: \"আমার বাবার বয়স ৬০ বছর। গত তিন মাস ধরে তার হাঁটুতে ব্যথা হচ্ছে। "
    "সিঁড়ি দিয়ে উঠতে গেলে বা বেশিক্ষণ হাঁটলে ব্যথা বেড়ে যায়। বিশ্রাম নিলে কিছুটা কমে। "
    "এক্স-রে করা হয়েছে, ডাক্তার বলেছেন হাড়ের ক্ষয় হয়েছে। কোনো ওষুধ এখনো শুরু হয়নি। "
    "কি করলে ভালো হবে?\"\n\n"
    "#Right Summary (RS)#: \"বয়স ৬০। তিন মাস ধরে হাঁটুতে ব্যথা, সিঁড়িতে বাড়ে, বিশ্রামে কমে। "
    "এক্স-রেতে হাড়ের ক্ষয় ধরা পড়েছে। চিকিৎসা শুরু হয়নি। করণীয় কী?\"\n\n"
    "#Hallucinated Summary (HS)#: \"বয়স ৬০। ছয় মাস ধরে দুই হাঁটুতে ব্যথা, হাঁটলে ও রাতে বাড়ে। "
    "এমআরআই-তে তরুণাস্থির ক্ষয় ধরা পড়েছে। চিকিৎসা শুরু হয়নি। করণীয় কী?\"\n\n"
    "Note: The HS changed 'তিন মাস' to 'ছয় মাস', added 'দুই হাঁটু' and 'রাতে বাড়ে' (not in source), "
    "and replaced 'এক্স-রে / হাড়ের ক্ষয়' with 'এমআরআই / তরুণাস্থির ক্ষয়' — all intrinsic distortions.\n\n"
    "Rules:\n"
    "- #Hallucinated Summary (HS)# length must be approximately equal to #Right Summary (RS)# length.\n"
    "- Do NOT add completely external facts; distort or over-infer from what is already in the source.\n"
    "- The hallucinated summary must still sound natural and plausible in Bangla.\n\n"
    "#Question (Q)#: <Here is the test question>\n"
    "#Right Summary (RS)#: <Here is the right summary of the test question>\n"
    "#Hallucinated Summary (HS)#: Generate"
 )

prompt_2 = (
    "I want you to act as a Hallucinated Summary (HS) generator. The answer should be given in BANGLA. "
    "Given a Question (Q) and the Right Summary (RS), your objective is to write a Hallucinated Summary (HS) "
    "that sounds plausible but is factually incorrect.\n\n"
    "You MUST use the following hallucination method:\n"
    "The summary states at least one concrete fact (a specific name, number, "
    "date, medicine name, place, or institution) that is entirely made up and does not appear anywhere "
    "in the source question.\n\n"
    "Example —\n"
    "#Question (Q)#: \"আমার মেয়ের বয়স ৮ বছর। গত কয়েকদিন ধরে তার জ্বর আসছে, সাথে গলা ব্যথাও আছে। "
    "জ্বর সর্বোচ্চ ১০২ ডিগ্রি পর্যন্ত উঠছে। প্যারাসিটামল দিচ্ছি, একটু কমে কিন্তু আবার আসে। "
    "খাওয়া-দাওয়া একদম কমে গেছে। কি করব?\"\n\n"
    "#Right Summary (RS)#: \"বয়স ৮। কয়েকদিন ধরে জ্বর ও গলা ব্যথা। জ্বর সর্বোচ্চ ১০২ ডিগ্রি। "
    "প্যারাসিটামল দেওয়া হচ্ছে। খাওয়া কমেছে। করণীয় কী?\"\n\n"
    "#Hallucinated Summary (HS)#: \"বয়স ৮। তিন সপ্তাহ ধরে জ্বর ও গলা ব্যথা। জ্বর সর্বোচ্চ ১০৪ ডিগ্রি। "
    "নাপা সিরাপ ও অ্যামোক্সিসিলিন দেওয়া হচ্ছে। খাওয়া কমেছে। করণীয় কী?\"\n\n"
    "Note: The Hallucinated Summary invented 'তিন সপ্তাহ' (source says 'কয়েকদিন'), changed '১০২' to '১০৪', "
    "and added 'অ্যামোক্সিসিলিন' — a drug name that does not appear anywhere in the source.\n\n"
    "Rules:\n"
    "- #Hallucinated Summary (HS)# length must be approximately equal to #Right Summary (RS)# length.\n"
    "- Do NOT add completely external facts; distort or over-infer from what is already in the source.\n"
    "- The hallucinated summary must still sound natural and plausible in Bangla.\n"
    "- At least one fabricated concrete fact (number, medicine, place, name, date) must be present.\n\n"
    "#Question (Q)#: <Here is the test question>\n"
    "#Right Summary (RS)#: <Here is the right summary of the test question>\n"
    "#Hallucinated Summary (HS)#: Generate"
 )

prompt_3 = (
    "I want you to act as a Hallucinated Summary (HS) generator. The answer should be given in BANGLA. "
    "Given a Question (Q) and the Right Summary (RS), your objective is to write a Hallucinated Summary (HS) "
    "that sounds plausible but is factually incorrect.\n\n"
    "You MUST use the following hallucination method:\n"
    "The summary directly contradicts at least one fact that is explicitly "
    "stated in the source question. This includes reversing a stated condition, changing a clear "
    "symptom to its opposite, or contradicting a specific detail the source makes unambiguous.\n\n"
    "Example —\n"
    "#Question (Q)#: \"আমার স্বামীর বয়স ৪৫ বছর। প্রায় দুই সপ্তাহ ধরে রাতে ঘুম হচ্ছে না। দিনের বেলায় "
    "ঘুম পায়, কিন্তু রাতে শুলেই মাথায় নানা চিন্তা আসে। কোনো ওষুধ খাচ্ছেন না। আগে এরকম হয়নি। "
    "কি করলে ঘুম ঠিক হবে?\"\n\n"
    "#Right Summary (RS)#: \"বয়স ৪৫। দুই সপ্তাহ ধরে রাতে ঘুম হচ্ছে না। দিনে ঘুম পায়। "
    "কোনো ওষুধ নেই। ঘুমের সমস্যার সমাধান জানতে চান।\"\n\n"
    "#Hallucinated Summary (HS)#: \"বয়স ৪৫। দুই সপ্তাহ ধরে দিনে ঘুম হচ্ছে না, রাতে অতিরিক্ত ঘুম পায়। "
    "ঘুমের ওষুধ সেবন করছেন। ঘুমের সমস্যার সমাধান জানতে চান।\"\n\n"
    "Note: The Hallucinated Summary reversed the sleep pattern ('রাতে ঘুম নেই' became 'রাতে অতিরিক্ত ঘুম'), "
    "and contradicted 'কোনো ওষুধ নেই' by stating 'ঘুমের ওষুধ সেবন করছেন' — both direct contradictions "
    "of explicitly stated facts in the source.\n\n"
    "Rules:\n"
    "- #Hallucinated Summary (HS)# length must be approximately equal to #Right Summary (RS)# length.\n"
    "- Do NOT add completely external facts; distort or over-infer from what is already in the source.\n"
    "- The hallucinated summary must still sound natural and plausible in Bangla.\n"
    "- At least one statement must directly contradict an explicit fact from the source question.\n"
    "- The contradiction should be subtle enough to still sound plausible in Bangla.\n\n"
    "#Question (Q)#: <Here is the test question>\n"
    "#Right Summary (RS)#: <Here is the right summary of the test question>\n"
    "#Hallucinated Summary (HS)#: Generate"
 )

PROMPT_MAP = {
    "Intrinsic": prompt_1,
    "Non-factual": prompt_2,
    "Factual Contradiction": prompt_3,
}


In [ ]:
if not os.environ.get('OPENAI_API_KEY') or os.environ['OPENAI_API_KEY'].strip() == '':
    raise RuntimeError('OPENAI_API_KEY is empty. Paste your key in the config cell above.')

LOG_PATH = OUTPUT_CSV.replace('.csv', '.log')

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

rows = load_rows(INPUT_CSV)
output_rows = []
completed_source_ids = set()
fieldnames = [
    'id',
    'source_id',
    'pattern',
    'document',
    'right_summary',
    'hallucinated_summary',
 ]

if Path(OUTPUT_CSV).exists():
    output_rows = load_rows(OUTPUT_CSV)
    completed_source_ids = {row.get('source_id') for row in output_rows if row.get('source_id')}
    print(f'Resuming with {len(completed_source_ids)} source items already completed')

with open(LOG_PATH, 'a', encoding='utf-8') as log:
    log.write('Starting summarization hallucination generation.\n')
    log.write(f'Total items: {len(rows)}\n')

for idx, row in enumerate(rows, start=1):
    source_id = (row.get('id') or '').strip()
    if source_id in completed_source_ids:
        continue
    document = (row.get(DOCUMENT_COLUMN) or '').strip()
    right_summary = (row.get(SUMMARY_COLUMN) or '').strip()

    for pattern_key, template in PROMPT_MAP.items():
        prompt = build_prompt(template, document, right_summary)
        response = client.responses.create(
            model=MODEL,
            input=[{'role': 'user', 'content': prompt}],
            max_output_tokens=96,
            temperature=0.7,
        )
        hallucinated = normalize_summary(response.output_text)
        output_rows.append({
            'id': f"{source_id}::{pattern_key}",
            'source_id': source_id,
            'pattern': pattern_key,
            'document': document,
            'right_summary': right_summary,
            'hallucinated_summary': hallucinated,
        })

    write_rows(OUTPUT_CSV, output_rows, fieldnames)
    with open(LOG_PATH, 'a', encoding='utf-8') as log:
        log.write(f'[{idx}/{len(rows)}] id={source_id} wrote {len(output_rows)} rows\n')
    print(f"Saved {len(output_rows)} rows after item {idx}")

print(f"Done. Wrote {len(output_rows)} rows to {OUTPUT_CSV}")
print(f"Log saved to {LOG_PATH}")
